# Figure 4: Intrinsic Spinal Gradient Organization - 8 ROI

This notebook reproduces **Figure 4** from the manuscript. It assumes precomputed timecourses and connectivity matrices are available on disk, as configured via `config-results.json`. Anatomical parcellation used :  48 spinal cord ROIs - Spanning C3–C8 spinal levels - 6 levels with 8 ROI each - PAM50 Atlas, are used for constructing the corticospinal FC matrix.


## Config and imports

This cell loads the configuration file and imports all required libraries and utility functions.

In [ ]:
from utils import *

# Load analysis config
params = read_config('config-results.json')

n_rois = 400
schaefer_dataset = datasets.fetch_atlas_schaefer_2018(n_rois=n_rois, resolution_mm=2)
schaefer_atlas = schaefer_dataset.maps
schaefer_labels = (schaefer_dataset.labels).astype(str)

sc_data = load_img(params["custom_sc_atlas"]).get_fdata()
sc_labels = open(params["custom_sc_labels"],'r').read().splitlines()

## Panel 4B

This cell loads the cortcospinal FC with SMC rois and 8-ROI parcellation of spinal cord. 
Computes mean FC, performs sparsification. 
Visualizes 8-ROI paracellation spinal only FC. 
Computing Gradients, Reference, Alignment, Variance explained, Gradient maps for 8-ROI case

In [ ]:
fc_files = sorted(
    glob.glob(os.path.join(params["save_fc_mats_8sp"], "*_subFC_8sp.csv"))
)

if len(fc_files) == 0:
    raise FileNotFoundError(
        f"No FC csv files found in: {params['save_fc_mats_8sp']}"
    )

subFC_mats = []
sub_rois = None

for fpath in fc_files:
    df_fc = pd.read_csv(fpath, index_col=0)

    # Check that each loaded FC matrix is square and labelled consistently
    if df_fc.shape[0] != df_fc.shape[1]:
        raise ValueError(
            f"Non-square FC matrix found in file: {fpath} with shape {df_fc.shape}"
        )

    if df_fc.index.tolist() != df_fc.columns.tolist():
        raise ValueError(
            f"Row/column ROI labels do not match in file: {fpath}"
        )

    # Store ROI order from the first file and enforce consistency across subjects
    if sub_rois is None:
        sub_rois = df_fc.index.tolist()
    else:
        if df_fc.index.tolist() != sub_rois:
            raise ValueError(
                f"ROI ordering mismatch in file: {fpath}"
            )

    subFC_mats.append(df_fc.to_numpy(dtype=float))

# Convert to array: n_subjects x n_rois x n_rois
subFC_mats = np.stack(subFC_mats, axis=0)

# ------------------------------------------------------------------
# Group-average sub-FC across subjects
# ------------------------------------------------------------------
mean_FC = np.mean(subFC_mats, axis=0)
subFC = mean_FC.copy()

# ------------------------------------------------------------------
# Rebuild ROI mapping from saved FC matrices
# ------------------------------------------------------------------
rois_incl = sub_rois
rois_map = [
    "LH_SomMot" if "LH_SomMot" in roi
    else "RH_SomMot" if "RH_SomMot" in roi
    else roi.split(" ")[0]
    for roi in rois_incl
]

# Identify cortical SMC indices within rois_incl
sm_idx = [j for j, roi in enumerate(rois_map) if "SomMot" in roi]

# ------------------------------------------------------------------
# Extract cortical SMC-SMC block
# ------------------------------------------------------------------
cortical_FC = subFC[np.ix_(sm_idx, sm_idx)]

# ------------------------------------------------------------------
# Sparsify cortical FC by retaining strongest edges per row
# ------------------------------------------------------------------
sparsity_cortical = 0.9
cortical_sparse = np.array([
    row * (row > np.sort(row)[int(sparsity_cortical * len(row)) - 1])
    for row in cortical_FC
])

# ------------------------------------------------------------------
# Replace cortical block in the full subFC to obtain corticospinal FC
# ------------------------------------------------------------------
FC_cs = subFC.copy()
FC_cs[np.ix_(sm_idx, sm_idx)] = cortical_sparse

#### Visualize Spinal FC

In [ ]:
# cortical_FC is your square connectivity matrix
spinal_FC = subFC_spinal = subFC[62:, 62:]
# 1) Z-score and clip to [-1, 1]
spinal_z = (spinal_FC - spinal_FC.mean()) / spinal_FC.std()
spinal_z = np.clip(spinal_z, -1, 1)

# 2) CSS-like diverging colormap
colors = ["#00a2ff", "#9ddff5", "#ffffff", "#ffbfdf", "#ff369b"]
cmap_css = LinearSegmentedColormap.from_list("css_fc_div", colors, N=256)

# 3) Figure size so each cell is ~square
n = spinal_z.shape[0]
cell_size = 0.1  # inches per cell
fig_size = n * cell_size

fig, ax = plt.subplots(figsize=(fig_size, fig_size), dpi=300)

sns.heatmap(
    spinal_z,
    cmap=cmap_css,
    square=True,          # enforce square cells
    cbar=True,
    linewidths=0.25,       # thin borders
    linecolor='white',
    xticklabels=False,
    yticklabels=False,
    ax=ax
)

cbar = ax.collections[0].colorbar
cbar.set_label("Z-scored FC", rotation=270, labelpad=15)

plt.tight_layout()
plt.show()
out_path = os.path.join(
    params["save_main_sc"],
    "figure4_FC_SC_8ROI.png"
)
fig.savefig(out_path, dpi=300, bbox_inches="tight")

#### Reference axes for Procrustes alignment
Axis 1: Ascending WM & Dorsal horns (-1) --> Intermediate Zone (0) --> Descending WM & Ventral horns (1).
Axis 2: Right GM (-1) --> Ascending & descending WM (0) --> Left GM (1)

In [ ]:
# select the group of RoIs on which to project the gradients to - i.e. all spinal RoIs at level C6
rois_spinal_x = [roi for j,roi in enumerate(rois_incl) if rois_map[j]=='C6']
# for corticospinal gradients, the y dimension of FC includes both C6 RoIs as well as SomMot RoIs
rois_corticospinal_y = [roi for j,roi in enumerate(rois_incl) if rois_map[j]=='C6'] + [roi for j,roi in enumerate(rois_incl) if 'SomMot' in roi]

# Reference axes for alignment with Procrustes
# Axis 1: Ascending WM & Dorsal horns (-1) --> Intermediate Zone (0) --> Descending WM & Ventral horns (1)
# Axis 2: Right GM (-1) --> Ascending & descending WM (0) --> Left GM (1)

ref_mat_custom = np.zeros((len(rois_spinal_x),2))
for j,roi in enumerate(rois_spinal_x):
    if roi.split(' ')[1] == 'WM':
        if roi.split(' ')[2] == 'descending':
            ref_mat_custom[j,0] = 1
        else:
            ref_mat_custom[j,0] = -1
    else:
        if roi.split(' ')[3] == 'ventral':
            ref_mat_custom[j,0] = 1
        elif roi.split(' ')[3] == 'dorsal':
            ref_mat_custom[j,0] = -1
        if roi.split(' ')[2] == 'left':
            ref_mat_custom[j,1] = 1
        else:
            ref_mat_custom[j,1] = -1           

#### Computing Gradients (spinal and corticospinal) + Alignment

The saved gradients maps are visualized with a specialized spinal map rendering on the local system. The results are presented in Panel 4B.

In [ ]:
### [spinal]
grads_spinal, lambdas_spinal, FC_spinal = fit_gradients(FC_cs, rois_spinal_x, rois_incl,
        n_components = 5, approach = 'dm', kernel = 'cosine', sparsity = 0)
ref_norm, grads_spinal_aligned, disparity_sc = align_procrustes_matlab(grads_spinal, ref_mat_custom, align_dims=2)

grads_spinal_aligned = (grads_spinal_aligned - np.mean(grads_spinal_aligned, axis=0))/np.std(grads_spinal_aligned, axis=0)

# _, grads_spinal_img = project_4D(grads_spinal_aligned, rois_spinal_x, ['C6']*len(rois_spinal_x), params)
# grads_spinal_img.to_filename(params['save_grads_sc'] + 'CS_spinal_C6_resting_aligned.nii.gz')

### [spinal-smc]
grads_corticospinal, lambdas_corticospinal, FC_corticospinal = fit_gradients(FC_cs, rois_spinal_x, rois_incl, rois_corticospinal_y, 
        n_components = 5, approach = 'dm', kernel = 'cosine', sparsity = 0)

ref_norm, grads_corticospinal_aligned, disparity_sc_cs = align_procrustes_matlab(grads_corticospinal, ref_mat_custom, align_dims=2)
grads_corticospinal_aligned = (grads_corticospinal_aligned - np.mean(grads_corticospinal_aligned, axis=0))/np.std(grads_corticospinal_aligned, axis=0)

#_ , grads_corticospinal_img = project_4D(grads_corticospinal_aligned, rois_spinal_x, ['C6']*len(rois_spinal_x), params)
#grads_corticospinal_img.to_filename(params['save_grads_sc'] + 'CS_corticospinal_C6_resting_aligned.nii.gz')

### Print explained variance & metrics (unchanged)
print("Explained variance - Spinal:")
print(lambdas_spinal)
print("\nExplained variance - Spinal-SMC:")
print(lambdas_corticospinal)
print(f"\nDisparity spinal aligned: {disparity_sc:.3f}")
print(f"Disparity spinal+cs aligned: {disparity_sc_cs:.3f}")
print(f"\nSpinal corr(G1,G2): {np.corrcoef(grads_spinal_aligned[:,0], grads_spinal_aligned[:,1])[0,1]:.3f}")
print(f"Spinal-SMC corr(G1,G2): {np.corrcoef(grads_corticospinal_aligned[:,0], grads_corticospinal_aligned[:,1])[0,1]:.3f}")

#### Explained variance

In [ ]:
x = np.arange(1, len(lambdas_spinal) + 1)
fig, ax = plt.subplots(figsize=(3, 4), dpi=300)
# Connected dots and lines
ax.plot(x, lambdas_spinal, marker='o', color='k', linewidth=1)
ax.set_xlabel('Component')
ax.set_ylabel('Explained variance (%)')
# Shaded block along x-axis from component 1 to 2
ax.axvspan(0.95, 2.05, color='grey', alpha=0.1)
ax.set_xticks(x)
ax.set_ylim(0, max(lambdas_spinal) * 1.1)
plt.tight_layout()
plt.show()
out_path = os.path.join(
    params["save_main_sc"],
    "figure4_SC_Gradients_ExpVar_8ROI.png"
)
fig.savefig(out_path, dpi=300, bbox_inches="tight")


## Panel 4C (Top)

This cell computes the scatter plots for 8-ROI case (spinal-spinal gradient space).

In [ ]:
# -------------------------------------------------------------------
# Config
# -------------------------------------------------------------------
roi_colors = {
    'GM left dorsal horn'        : '#0072B2',  # blue
    'GM left intermediate zone'  : '#009E73',  # green
    'GM left ventral horn'       : '#E69F00',  # orange
    'WM descending'              : '#D55E00',  # vermillion
    'GM right ventral horn'      : '#CC79A7',  # purple
    'GM right intermediate zone' : '#56B4E9',  # light blue
    'GM right dorsal horn'       : '#F0E442',  # yellow
    'WM ascending'               : '#000000'   # black
}

rois_order_template = [
    'GM left dorsal horn',
    'GM left intermediate zone',
    'GM left ventral horn',
    'WM descending',
    'GM right ventral horn',
    'GM right intermediate zone',
    'GM right dorsal horn',
    'WM ascending'
]

# -------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------

def get_spinal_rois_for_level(level, rois_incl, rois_map):
    """
    level: 'C4'..'C8'
    returns:
        rois_spinal_x (list of full labels, e.g. 'C6 GM left dorsal horn')
        rois_corticospinal_y (C-level + SomMot)
    """
    rois_spinal_x = [roi for j, roi in enumerate(rois_incl) if rois_map[j] == level]
    rois_corticospinal_y = rois_spinal_x + [roi for roi in rois_incl if 'SomMot' in roi]
    return rois_spinal_x, rois_corticospinal_y


def reorder_gradients_by_level(level, grad_matrix, rois_spinal_x, rois_order_template):
    """
    grad_matrix: (8 x 2) gradients for this level, in order rois_spinal_x
    returns:
        (8 x 2) in anatomical plot order
    """
    plot_order_full = [f'{level} {roi}' for roi in rois_order_template]
    reordered = np.array([grad_matrix[rois_spinal_x.index(roi), :] for roi in plot_order_full])
    return reordered, plot_order_full


def compute_subject_gradients_for_level(FC_mat, level, rois_incl, rois_map, rois_list,
                                        ref_mat_custom, n_components=10,
                                        clip_val=None, mode='both'):
    """
    Compute aligned gradients (spinal and/or spinal-SMC) for a single subject and spinal level.
    mode: 'spinal', 'corticospinal', or 'both'
    clip_val: if not None, clip gradients to [-clip_val, clip_val]
    Returns:
        grads_spinal_reordered or None: (8 x 2)
        grads_corticospinal_reordered or None: (8 x 2)
        plot_order_full
    """
    rois_spinal_x, rois_corticospinal_y = get_spinal_rois_for_level(level, rois_incl, rois_map)

    # Index for this subject's FC
    idx_y = [rois_list.index(roi) for roi in rois_corticospinal_y]
    FC_incl = FC_mat[np.ix_(idx_y, idx_y)]

    grads_spinal_reordered = None
    grads_corticospinal_reordered = None

    # Spinal-only gradients (x = spinal, y = all corticospinal_y)
    if mode in ('spinal', 'both'):
        grads_spinal, _, _ = fit_gradients(
            FC_incl, rois_spinal_x, rois_corticospinal_y,
            n_components=n_components, approach='dm', kernel='cosine', sparsity=0
        )
        _, grads_spinal_aligned, _ = align_procrustes_matlab(grads_spinal, ref_mat_custom, align_dims=2)
        grads_spinal_aligned = (grads_spinal_aligned - np.mean(grads_spinal_aligned, axis=0)) / np.std(grads_spinal_aligned, axis=0)
        
        if clip_val is not None:
            grads_spinal_aligned = np.clip(grads_spinal_aligned, -clip_val, clip_val)

        grads_spinal_reordered, plot_order_full = reorder_gradients_by_level(
            level, grads_spinal_aligned, rois_spinal_x, rois_order_template
        )
    else:
        # needed for return; still define plot_order_full for consistency
        plot_order_full = [f'{level} {roi}' for roi in rois_order_template]

    # Corticospinal gradients (x = spinal, y = spinal+SomMot)
    if mode in ('corticospinal', 'both'):
        grads_corticospinal, _, _ = fit_gradients(
            FC_incl, rois_spinal_x, rois_corticospinal_y, rois_corticospinal_y,
            n_components=n_components, approach='dm', kernel='cosine', sparsity=0
        )
        _, grads_corticospinal_aligned, _ = align_procrustes_matlab(grads_corticospinal, ref_mat_custom, align_dims=2)
        grads_corticospinal_aligned = (grads_corticospinal_aligned - np.mean(grads_corticospinal_aligned, axis=0)) / np.std(grads_corticospinal_aligned, axis=0)
        if clip_val is not None:
            grads_corticospinal_aligned = np.clip(grads_corticospinal_aligned, -clip_val, clip_val)

        grads_corticospinal_reordered, _ = reorder_gradients_by_level(
            level, grads_corticospinal_aligned, rois_spinal_x, rois_order_template
        )

    return grads_spinal_reordered, grads_corticospinal_reordered, plot_order_full


def compute_centroids(points_per_roi):
    """
    points_per_roi: dict[roi_name] -> list of (N_i x 2) points from all subjects
    returns: dict[roi_name] -> (2,) centroid
    """
    centroids = {}
    for roi, pts in points_per_roi.items():
        pts_arr = np.vstack(pts)  # (N_total x 2)
        centroids[roi] = pts_arr.mean(axis=0)
    return centroids


# -------------------------------------------------------------------
# Main subject-wise plotting function
# -------------------------------------------------------------------
subject_list = [f'S{i:02d}' for i in range(1, 21) if i != 15]
def plot_subject_gradients_with_centroids(FC_mats, level,
                                          rois_incl, rois_map, rois_list,
                                          ref_mat_custom,
                                          save_dir, prefix='C_level',
                                          clip_val=None,
                                          mode='both',
                                          max_subjects=None,
                                          subject_ids=None,
                                          select_subject=None):
    """
    ...
    subject_ids: optional list of IDs matching FC_mats (e.g. ['S01', 'S02', ...])
    select_subject: optional subject ID to plot only that subject (e.g. 'S04')
    """
    # Optionally select a specific subject OR restrict number of subjects
    if select_subject is not None and subject_ids is not None:
        try:
            idx = subject_ids.index(select_subject)
        except ValueError:
            raise ValueError(f"Subject {select_subject} not found in subject_ids")
        FC_mats_use = [FC_mats[idx]]
    else:
        if max_subjects is not None:
            FC_mats_use = FC_mats[:max_subjects]
        else:
            FC_mats_use = FC_mats

    n_subj_used = len(FC_mats_use)

    # Containers: per ROI, stack all subject points
    spinal_points = {roi: [] for roi in rois_order_template}
    corticospinal_points = {roi: [] for roi in rois_order_template}

    # NEW: arrays to store per-subject gradients (8 x 2)
    grads_spinal_all = np.zeros((n_subj_used, len(rois_order_template), 2), dtype=float) if mode in ('spinal', 'both') else None
    grads_corticospinal_all = np.zeros((n_subj_used, len(rois_order_template), 2), dtype=float) if mode in ('corticospinal', 'both') else None


    # Loop over subjects
    for s, FC_mat in enumerate(FC_mats_use):
        g_spinal, g_cs, plot_order_full = compute_subject_gradients_for_level(
            FC_mat, level, rois_incl, rois_map, rois_list, ref_mat_custom,
            n_components=10, clip_val=clip_val, mode=mode
        )

        # g_spinal, g_cs: (8 x 2) in anatomical order matching rois_order_template
        if mode in ('spinal', 'both') and g_spinal is not None:
            grads_spinal_all[s, :, :] = g_spinal
            for i, roi_base in enumerate(rois_order_template):
                spinal_points[roi_base].append(g_spinal[i, :])

        if mode in ('corticospinal', 'both') and g_cs is not None:
            grads_corticospinal_all[s, :, :] = g_cs
            for i, roi_base in enumerate(rois_order_template):
                corticospinal_points[roi_base].append(g_cs[i, :])
    # Compute centroids
    spinal_centroids = compute_centroids(spinal_points) if mode in ('spinal', 'both') else {}
    corticospinal_centroids = compute_centroids(corticospinal_points) if mode in ('corticospinal', 'both') else {}

    # ----------------------------------------------------------------
    # Figure 1: scatter + centroids
    # ----------------------------------------------------------------
    plt.figure(figsize=(8, 10), dpi=300)

    # Scatter: all subjects, both conditions as requested
    for roi_base in rois_order_template:
        color = roi_colors[roi_base]

        if mode in ('spinal', 'both') and len(spinal_points[roi_base]) > 0:
            pts_sp = np.vstack(spinal_points[roi_base])
            plt.scatter(
                pts_sp[:, 1], pts_sp[:, 0],
                s=200, alpha=0.5,
                facecolor=color, edgecolor='black', linewidth=0.5,
                label=None
            )

        if mode in ('corticospinal', 'both') and len(corticospinal_points[roi_base]) > 0:
            pts_cs = np.vstack(corticospinal_points[roi_base])
            plt.scatter(
                pts_cs[:, 1], pts_cs[:, 0],
                s=200, alpha=0.5,
                facecolor='none', edgecolor=color, linewidth=2.0,
                label=None
            )

    # Centroids: larger markers
    for roi_base in rois_order_template:
        color = roi_colors[roi_base]

        if mode in ('spinal', 'both') and roi_base in spinal_centroids:
            cx_sp, cy_sp = spinal_centroids[roi_base][1], spinal_centroids[roi_base][0]
            plt.scatter(
                cx_sp, cy_sp,
                s=400, facecolor=color, edgecolor='black', linewidth=1.5,
                marker='o', label=None
            )

        if mode in ('corticospinal', 'both') and roi_base in corticospinal_centroids:
            cx_cs, cy_cs = corticospinal_centroids[roi_base][1], corticospinal_centroids[roi_base][0]
            plt.scatter(
                cx_cs, cy_cs,
                s=400, facecolor='white', edgecolor=color, linewidth=5.0,
                marker='o', label=None
            )

    plt.xlabel('G2', fontsize=12, fontweight='bold')
    plt.ylabel('G1', fontsize=12, fontweight='bold')

    title_clip = f' (clipped to ±{clip_val})' if clip_val is not None else ''
    plt.title(
        f'{level} spinal gradients across subjects\nScatter + centroids, mode={mode}{title_clip}',
        fontsize=14, fontweight='bold'
    )
    #plt.grid(True, alpha=0.3)

    # Legend: one entry per ROI, as before
    legend_handles = []
    legend_labels = []
    for roi_base in rois_order_template:
        color = roi_colors[roi_base]
        h = plt.Line2D(
            [0], [0],
            marker='o',
            color='w',
            markerfacecolor=color,
            markeredgecolor='black',
            markersize=8,
            linewidth=0
        )
        legend_handles.append(h)
        legend_labels.append(roi_base)
    plt.legend(legend_handles, legend_labels, loc='best', bbox_to_anchor=(1, 0.5), frameon=True, title='ROIs')

    plt.tight_layout()
    plt.savefig(f'{save_dir}{prefix}_{level}_scatter_centroids_8_mode-{mode}.png', dpi=300, bbox_inches='tight')
    plt.show()
    return grads_spinal_all, grads_corticospinal_all


##### Group Level (all subjects)

In [ ]:
grads_spinal_all, grads_corticospinal_all = plot_subject_gradients_with_centroids(
    FC_mats=subFC_mats,
    level='C6',
    rois_incl=rois_incl,
    rois_map=rois_map,
    rois_list=sub_rois,
    ref_mat_custom=ref_mat_custom,
    save_dir=params['save_grads_sc_scatter'],
    prefix='figure4_spinal_gradients_group',
    clip_val=2.0,
    mode='spinal',
    subject_ids=subject_list,
    select_subject=None
)


##### Single Subject

In [ ]:
grads_spinal_all, grads_corticospinal_all = plot_subject_gradients_with_centroids(
    FC_mats=subFC_mats,
    level='C6',
    rois_incl=rois_incl,
    rois_map=rois_map,
    rois_list=sub_rois,
    ref_mat_custom=ref_mat_custom,
    save_dir=params['save_grads_sc_scatter'],
    prefix='figure4_spinal_gradients_single',
    clip_val=2.5,
    mode='spinal',
    subject_ids='S05',
    select_subject='S05'
)
